Importation des bibliothèques

In [1]:
import os
import random
# import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import random_split
from tqdm import tqdm # pour la progression 
import time #pour le temps de calcul
import copy #pour copier base_model

In [2]:
print(os.getcwd())  # donne le répertoire courant

d:\INRIA\MCDropout


Fixation du seed pour la reproductibilité

In [3]:
seed = 42
random.seed(seed)
# np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # si GPU dispo

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Configuration de base

Importation du modèle déjà entraîné par l'utilisateur

In [4]:
batch_size = 128 #on peut changer la taille mais 128 est bien pour éviter underfitting ou overfitting

transform = transforms.Compose([
    transforms.ToTensor(),                 #Convertir une image en tenseur, met les valeurs des pixels entre 0 et 1
    transforms.Normalize((0.5, 0.5, 0.5),  # Moyenne RGB
                         (0.5, 0.5, 0.5))  # Écart-type RGB; pixels deviennent centrés autour de 0
])


trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')


Définition du CNN de base (3 couches)

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # entrée 3 channels (RGB)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)  # réduit dimension par 2 à chaque fois
        
        # Calculer la taille en sortie avant les couches fully connected
        # CIFAR10 32x32 → après 3 poolings (x2) → 4x4
        self.fc1 = nn.Linear(64 * 4 * 4, 128) # après les convolutions on a un tenseur de taille 64*4*4, fc1 les réduit en 128 neurones
        self.fc2 = nn.Linear(128, 10)  # 10 classes CIFAR10

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        
        x = x.view(x.size(0), -1)  # aplatit en [batch_size, 64*4*4 = 1024]
        x = F.relu(self.fc1(x)) #ou fonction sigmoïde à la fin?
        x = self.fc2(x)
        return x


à entraîner (boucle d'entraînement) training loop + sauvegarde des poids à chaque epoch ; tester pour chaque epoch et voir où l'accuracy est la mieux

Masque pour le dropout

In [6]:
class CNN_MCdropout(nn.Module):
    def __init__(self, model, mc_layers=None, p1=0.2, p2=0.2, p3=0.2, p4=0.2):
        super().__init__()
        self.model = model
        self.mc_layers = mc_layers or [] # si None, aucune couche à masquer
        self.p1 = p1  # dropout sur conv1 output
        self.p2 = p2  # dropout sur conv2 output
        self.p3 = p3  # dropout sur conv3 output
        self.p4 = p4  # dropout sur fc1 output
        # pas de dropout sur la dernière couche
        self.ps = {'conv1': p1, 'conv2': p2, 'conv3': p3, 'fc1': p4}
    
    def dropout_mask(self, x, p, active=True):
        if not self.training or p == 0.0 or not active:
            return torch.ones_like(x) #conserve la dimension de la couche
        mask = (torch.rand_like(x) > p).float() / (1 - p)
        return mask

    def forward(self, x):
        x = F.relu(self.model.conv1(x))
        x = self.dropout_mask(x, self.ps['conv1'], 'conv1' in self.mc_layers) * x
        x = self.model.pool(x)

        x = F.relu(self.model.conv2(x))
        x = self.dropout_mask(x, self.ps['conv2'], 'conv2' in self.mc_layers) * x
        x = self.model.pool(x)

        x = F.relu(self.model.conv3(x))
        x = self.dropout_mask(x, self.ps['conv3'], 'conv3' in self.mc_layers) * x
        x = self.model.pool(x)

        x = x.view(x.size(0), -1) 
        x = F.relu(self.model.fc1(x))
        x = self.dropout_mask(x, self.ps['fc1'], 'fc1' in self.mc_layers) * x
        x = self.model.fc2(x)
        return x  # pas encore normalisé
    
    # x = x.view ... 
    # x = self.dropout_mask
    # x = F.relu(self.model.fc1(x))
    # x = self.model.fc2(x)

    # faire pareil pour chaque couche concernée, même celles pas concernées, faire le dropout avant relu (pour faire comme pytorch, masque les arêtes entrantes)
    # dans le nn.sequential, prend effet que si model.train()

In [7]:
#Avant ReLU
class CNN_MCdropout_beforeReLU(nn.Module):
    def __init__(self, model, mc_layers=None, p1=0.2, p2=0.2, p3=0.2, p4=0.2):
        super().__init__()
        self.model = model
        self.mc_layers = mc_layers or []
        self.ps = {'conv1': p1, 'conv2': p2, 'conv3': p3, 'fc1': p4}

    def dropout_mask(self, x, p, active=True):
        if not self.training or p == 0.0 or not active:
            return torch.ones_like(x)
        mask = (torch.rand_like(x) > p).float() / (1 - p)
        return mask

    def forward(self, x):
        x = self.model.conv1(x)
        x = self.dropout_mask(x, self.ps['conv1'], 'conv1' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.pool(x)

        x = self.model.conv2(x)
        x = self.dropout_mask(x, self.ps['conv2'], 'conv2' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.pool(x)

        x = self.model.conv3(x)
        x = self.dropout_mask(x, self.ps['conv3'], 'conv3' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.pool(x)

        x = x.view(x.size(0), -1)
        x = self.model.fc1(x)
        x = self.dropout_mask(x, self.ps['fc1'], 'fc1' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.fc2(x)
        return x


In [8]:
class CNN_MCdropout_torch(nn.Module):
    def __init__(self, model, dico_layers=None, before=False):
        super().__init__()
        self.model = model
        self.dico_layers = dico_layers or {}
        self.before = before

        for name, layer in list(self.model._modules.items()):
            if isinstance(layer, nn.Conv2d) and name in self.dico_layers:
                p = self.dico_layers[name]
                if self.before:
                    self.model._modules[name] = nn.Sequential(nn.Dropout2d(p), layer)
                else:
                    self.model._modules[name] = nn.Sequential(layer, nn.Dropout2d(p))

            elif isinstance(layer, nn.Linear) and name in self.dico_layers:
                p = self.dico_layers[name]
                if self.before:
                    self.model._modules[name] = nn.Sequential(nn.Dropout(p), layer)
                else:
                    self.model._modules[name] = nn.Sequential(layer, nn.Dropout(p))
                
    def forward(self, x):
        return self.model.forward(x) 

Fonction d'entraînement et test

In [9]:
val_ratio = 0.1  # 10% pour validation
train_size = int((1 - val_ratio) * len(trainset))
val_size   = len(trainset) - train_size

# on fixe tjrs les mêmes train et val set
train_subset, val_subset = random_split(trainset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))

trainloader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader   = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)

Fonction d'évaluation (avant entraînement)

In [10]:
def evaluate(model, dataloader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss/len(dataloader), correct/total

Fonction d'entraînement

In [11]:
def train_model(model, trainloader, valloader, device, epochs=20, save_path="best.pt"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0
    
    for epoch in range(epochs):#petit nombre d'epochs pour tester (environ 1 minutes pas epoch)
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Évaluer sur validation
        val_loss, val_acc = evaluate(model, valloader, device)
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {running_loss/len(trainloader):.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")
        
        # Sauvegarde si amélioration
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
    
    print("Finished Training")
    return model

In [12]:
dico_layers = {"conv1": 0.2, "conv2": 0.2, "conv3": 0.2, "fc1": 0.2}

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best.pt"

# Demande à l'utilisateur quelles couches doivent subir le dropout : ou alors il amène sa liste
user_layers = input(
    "Sur quelles couches voulez-vous appliquer le MC Dropout ? "
    "(choisissez parmi conv1, conv2, conv3, fc1, séparées par des virgules) : ")
mc_layers = [layer.strip() for layer in user_layers.split(',') if layer.strip() in ['conv1','conv2','conv3','fc1']]

# Vérifie si les poids existent déjà
base_model = SimpleCNN()
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # même architecture que celle qui a sauvegardé
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = train_model(base_model, trainloader, valloader, device, epochs=20, save_path=save_path)
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # recharge les meilleurs poids

# On fait une copie pour MC Dropout
base_model_mc1 = copy.deepcopy(base_model)
base_model_mc2 = copy.deepcopy(base_model)
base_model_mc3 = copy.deepcopy(base_model)
base_model_mc4 = copy.deepcopy(base_model)

model = CNN_MCdropout(base_model_mc1, mc_layers=mc_layers, p1=0.2, p2=0.2, p3=0.2, p4=0.2).to(device)
model_beforeReLU = CNN_MCdropout_beforeReLU(base_model_mc2, mc_layers=mc_layers, p1=0.2, p2=0.2, p3=0.2, p4=0.2).to(device)
model_beforeTorch = CNN_MCdropout_torch(base_model_mc3, dico_layers=dico_layers, before=True).to(device)
model_afterTorch = CNN_MCdropout_torch(base_model_mc4, dico_layers=dico_layers, before=False).to(device)

# Évaluation finale
test_loss, test_acc = evaluate(model, testloader, device)
print(f"Final Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")
test_loss2, test_acc2 = evaluate(model_beforeReLU, testloader, device)
print(f"Final Test Loss: {test_loss2:.4f} - Test Acc: {test_acc2:.4f}")
test_loss3, test_acc3 = evaluate(model_beforeTorch, testloader, device)
print(f"Final Test Loss: {test_loss3:.4f} - Test Acc: {test_acc3:.4f}")
test_loss4, test_acc4 = evaluate(model_afterTorch, testloader, device)
print(f"Final Test Loss: {test_loss4:.4f} - Test Acc: {test_acc4:.4f}")

Chargement du modèle sauvegardé


TypeError: CNN_MCdropout.__init__() got an unexpected keyword argument 'dico_layers'

MC Dropout prédiction

In [ ]:
def mc_predict_mean_probs(model, X, T=1000, verbose=True):
    model.train()  # activer dropout pendant l'inférence MC
    probs_list = []
    start_time = time.time() 
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            logits_t = model(X)
            probs_t = F.softmax(logits_t, dim=1)
            probs_list.append(probs_t.unsqueeze(0))
    elapsed = time.time() - start_time
    probs_mc = torch.cat(probs_list, dim=0)       
    model.eval()  # remettre le modèle en mode eval à la fin
    if verbose:
        print(f"Temps total: {elapsed:.2f} s  |  Temps moyen par passe: {elapsed/T:.4f} s")
    return probs_mc.mean(0), elapsed                        

je dois garder le même batch ; mettre des seeds pour que ce soit reproductible (fonction de dropout)

In [ ]:
# Test MC Dropout sur un batch
X, Y = next(iter(testloader))
X, Y = X.to(device), Y.to(device)
probs, t1 = mc_predict_mean_probs(model, X, T=1000, verbose=True)
probs2, t2 = mc_predict_mean_probs(model_beforeReLU, X, T=1000, verbose=True)
probs3, t3 = mc_predict_mean_probs(model_beforeTorch, X, T=1000, verbose=True)
probs4, t4 = mc_predict_mean_probs(model_afterTorch, X, T=1000, verbose=True)
Y_hat = probs.argmax(1)
Y_hat2 = probs2.argmax(1)
Y_hat3 = probs3.argmax(1)
Y_hat4 = probs4.argmax(1)

print("Classes vraies       :", Y.tolist())
print(f"After ReLU   (t={t1:.2f}s): {Y_hat.tolist()}")
print(f"Before ReLU  (t={t2:.2f}s): {Y_hat2.tolist()}")
print(f"Torch Dropout before Layer (t={t3:.2f}s): {Y_hat3.tolist()}")
print(f"Torch Dropout After Layer (t={t4:.2f}s): {Y_hat4.tolist()}")

acc = (Y_hat == Y).float().mean().item()
print(f"MC Dropout After ReLU Test Acc: {acc:.4f}")
acc_bf = (Y_hat2 == Y).float().mean().item()
print(f"MC Dropout Before ReLU Test Acc: {acc_bf:.4f}")
acc_tc = (Y_hat3 == Y).float().mean().item()
print(f"MC Dropout Torch Before Layer Test Acc: {acc_tc:.4f}")
acc_ta = (Y_hat4 == Y).float().mean().item()
print(f"MC Dropout Torch After Layer Test Acc: {acc_ta:.4f}")


100%|██████████| 1000/1000 [00:39<00:00, 25.52it/s]


Temps total: 39.19 s  |  Temps moyen par passe: 0.0392 s


100%|██████████| 1000/1000 [00:59<00:00, 16.78it/s]


Temps total: 59.58 s  |  Temps moyen par passe: 0.0596 s


100%|██████████| 1000/1000 [00:35<00:00, 28.29it/s]

Temps total: 35.36 s  |  Temps moyen par passe: 0.0354 s
Classes vraies       : [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9, 5, 2, 4, 0, 9, 6, 6, 5, 4, 5, 9, 2, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9, 7, 6, 9, 8, 0, 3, 8, 8, 7, 7, 4, 6, 7, 3, 6, 3, 6, 2, 1, 2, 3, 7, 2, 6, 8, 8, 0, 2, 9, 3, 3, 8, 8, 1, 1, 7, 2, 5, 2, 7, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 6, 2, 1, 3, 0, 4, 2, 7, 8, 3, 1, 2, 8, 0, 8, 3]
After ReLU   (t=39.19s): [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 3, 8, 6, 7, 2, 4, 1, 4, 2, 4, 0, 9, 6, 3, 5, 2, 3, 9, 3, 4, 1, 9, 5, 4, 6, 3, 6, 0, 9, 3, 3, 7, 2, 9, 8, 6, 3, 8, 8, 5, 5, 5, 3, 7, 5, 6, 3, 6, 2, 1, 0, 3, 1, 0, 6, 8, 8, 0, 2, 2, 3, 3, 8, 8, 1, 1, 7, 2, 2, 2, 8, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 3, 4, 5, 6, 3, 1, 1, 2, 6, 3, 5, 5, 0, 2, 2, 1, 3, 0, 4, 3, 5, 8, 2, 1, 2, 8, 0, 8, 3]
Before ReLU  (t=59.58s): [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 2, 4, 1, 4, 2, 4, 0, 9, 6, 3, 5, 2, 

In [ ]:
def generate_mc_outputs(model, X, T=1000, metrics=["mc_estimate"], labels=None, verbose=True):
    """
    Effectue T passes Monte Carlo Dropout sur le modèle pour le batch X,
    calcule les métriques d'incertitude spécifiées.
    Retourne les sorties de chaque passe, la moyenne des probabilités softmax,
    un dictionnaire des valeurs de métriques, et les temps de calcul.
    """
    model.train()
    outputs = []
    mean_probs = None 

    start_forward = time.time()
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            out = model(X)
            outputs.append(out.unsqueeze(0))
    elapsed_forward = time.time() - start_forward

    outputs = torch.cat(outputs, dim=0)  # [T, batch, num_classes]
    results = {}
    elapsed_metrics = {}

    # Calcul de la classe prédite initialement (premier passage sans dropout)
    with torch.no_grad():
        first_logits = model(X)
        first_probs = torch.softmax(first_logits, dim=1)
        initial_pred = first_probs.argmax(dim=1)  # [batch], classe prédite pour chaque échantillon
        
    all_probs = torch.softmax(outputs, dim=2)  
    mean_probs = all_probs.mean(dim=0) 

    for metric in metrics:
        start_metric = time.time()

        if metric == "mc_estimate":
            results[metric] = mean_probs

        elif metric == "variance_predicted": # var des probas softmax de la classe prédite initialement
            idx = initial_pred.unsqueeze(0).expand(T, -1) # matrice avec T lignes égales à initial_pred
            selected_probs = all_probs.gather(2, idx.unsqueeze(2)).squeeze(2) # pour chaque batch, on prend la colonne de la classe prédite initialement
            var_pred_class = selected_probs.var(dim=0) 
            results["variance_predicted_mean"] = var_pred_class.mean().item()
            results["variance_predicted"] = var_pred_class

        elif metric == "variance_max": # variance des probas max (toutes classes confondues) 
            max_probs, _ = all_probs.max(dim=2)  # shape [T, batch]
            var_max = max_probs.var(dim=0)  # shape [batch]
            results["variance_max_mean"] = var_max.mean().item()
            results["variance_max"] = var_max

        elif metric == "predictive_entropy_predicted": # PE pour la classe prédite initialement      
            idx = initial_pred.unsqueeze(1)          
            selected_mean_probs = mean_probs.gather(1, idx).squeeze(1) # on sélectionne la proba moyenne de la classe prédite
            # entropie binaire pour la classe prédite (p*log(p) + (1-p)*log(1-p))
            entropies_pred = -(selected_mean_probs * (selected_mean_probs + 1e-12).log() +
                               (1 - selected_mean_probs) * ((1 - selected_mean_probs + 1e-12).log()))
            results["predictive_entropy_predicted_mean"] = entropies_pred.mean().item()
            results["predictive_entropy_predicted"] = entropies_pred

        elif metric == "predictive_entropy_max": # PE de la probabilité max (toutes classes confondues)
            max_probs, _ = mean_probs.max(dim=1)  # shape [batch]
            # entropie binaire associée à p_max
            entropies = -(max_probs * (max_probs + 1e-12).log() +
                          (1 - max_probs) * ((1 - max_probs + 1e-12).log()))
            results["predictive_entropy_max_mean"] = entropies.mean().item()
            results["predictive_entropy_max"] = entropies

        elif metric == "relative_norm":
            if labels is None:
                raise ValueError("labels doivent être fournis pour relative_norm")
            labels_onehot = F.one_hot(labels, num_classes=mean_probs.size(1)).float()
            diff_norm = torch.norm(mean_probs - labels_onehot, dim=1)
            denom = torch.max(torch.norm(mean_probs, dim=1), torch.norm(labels_onehot, dim=1))
            relative_norm = diff_norm / (denom + 1e-12)
            results[metric + "_mean"] = relative_norm.mean().item()
            results[metric] = relative_norm

        else:
            raise ValueError(f"Métrique {metric} non reconnue")

        elapsed_metrics[metric] = time.time() - start_metric

    model.eval()

    if verbose:
        print(f"Temps forward pass: {elapsed_forward:.2f}s  |  Temps moyen par passe: {elapsed_forward/T:.4f}s")
        for m, t in elapsed_metrics.items():
            print(f"Temps calcul métrique '{m}': {t:.6f}s")

    return outputs, mean_probs, results, elapsed_forward, elapsed_metrics


Métriques avec la méthode after ReLU

In [ ]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance_predicted, variance_max, predictive_entropy_predicted, predictive_entropy_max, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(
    model, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")

for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")

    metric_result = metric_values[metric]

    if metric == "variance_predicted":
        print(f"Variance softmax moyenne : {metric_values['variance_predicted_mean']:.6f}")
        print(f"Variance softmax par échantillon :\n{metric_result}\n")
    elif metric == "variance_max":
        print(f"Variance max moyenne : {metric_values['variance_max_mean']:.6f}")
        print(f"Variance max par échantillon :\n{metric_result}\n")
    elif metric == "predictive_entropy_predicted":
        print(f"Entropie prédictive moyenne : {metric_values['predictive_entropy_predicted_mean']:.6f}")
        print(f"Entropie prédictive par échantillon :\n{metric_result}\n")
    elif metric == "predictive_entropy_max":
        print(f"Entropie max moyenne : {metric_values['predictive_entropy_max_mean']:.6f}")
        print(f"Entropie max par échantillon :\n{metric_result}\n")
    elif metric == "relative_norm":
        print(f"Norme relative moyenne : {metric_values['relative_norm_mean']:.6f}")
        print(f"Norme relative par échantillon :\n{metric_result}\n")
    else:
        print(f"Résultat : {metric_result}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 41.15 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.004865 s
Résultat : tensor([[0.0635, 0.0428, 0.0775,  ..., 0.0176, 0.1697, 0.0362],
        [0.0900, 0.3514, 0.0233,  ..., 0.0059, 0.4395, 0.0322],
        [0.1717, 0.1451, 0.0747,  ..., 0.0409, 0.3052, 0.0964],
        ...,
        [0.3171, 0.1148, 0.1875,  ..., 0.0060, 0.1152, 0.0920],
        [0.2493, 0.0297, 0.1289,  ..., 0.0138, 0.2924, 0.0322],
        [0.0387, 0.0197, 0.0615,  ..., 0.0535, 0.0317, 0.0424]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.007057 s
Variance moyenne : 58.827477
Variance par échantillon :
tensor([[ 44.2421,  62.7429,  34.7786,  ...,  47.7202,  70.1431,  80.3528],
        [ 44.2741,  88.1240,  59.9268,  ..., 103.0225,  81.2063,  58.9220],
        [ 21.5629,  41.8432,  25.5731,  ...,  44.8464,  37.6730,  34.2369],
  

Métriques avec la méthode before ReLU

In [ ]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance_predicted, variance_max, predictive_entropy_predicted, predictive_entropy_max, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(
    model_beforeReLU, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")

for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")

    metric_result = metric_values[metric]

    if metric == "variance_predicted":
        print(f"Variance softmax moyenne : {metric_values['variance_predicted_mean']:.6f}")
        print(f"Variance softmax par échantillon :\n{metric_result}\n")
    elif metric == "variance_max":
        print(f"Variance max moyenne : {metric_values['variance_max_mean']:.6f}")
        print(f"Variance max par échantillon :\n{metric_result}\n")
    elif metric == "predictive_entropy_predicted":
        print(f"Entropie prédictive moyenne : {metric_values['predictive_entropy_predicted_mean']:.6f}")
        print(f"Entropie prédictive par échantillon :\n{metric_result}\n")
    elif metric == "predictive_entropy_max":
        print(f"Entropie max moyenne : {metric_values['predictive_entropy_max_mean']:.6f}")
        print(f"Entropie max par échantillon :\n{metric_result}\n")
    elif metric == "relative_norm":
        print(f"Norme relative moyenne : {metric_values['relative_norm_mean']:.6f}")
        print(f"Norme relative par échantillon :\n{metric_result}\n")
    else:
        print(f"Résultat : {metric_result}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 45.33 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.017686 s
Résultat : tensor([[0.0762, 0.0446, 0.0665,  ..., 0.0180, 0.1564, 0.0316],
        [0.0874, 0.3780, 0.0124,  ..., 0.0038, 0.4259, 0.0348],
        [0.1827, 0.1431, 0.0686,  ..., 0.0391, 0.3139, 0.0804],
        ...,
        [0.2989, 0.1381, 0.1900,  ..., 0.0076, 0.1119, 0.0887],
        [0.2385, 0.0339, 0.1388,  ..., 0.0126, 0.2733, 0.0199],
        [0.0448, 0.0247, 0.0659,  ..., 0.0478, 0.0264, 0.0303]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.012789 s
Variance moyenne : 58.687584
Variance par échantillon :
tensor([[ 43.7381,  55.2870,  29.7358,  ...,  44.7788,  59.6763,  77.0072],
        [ 43.6900,  82.9875,  53.1798,  ..., 103.2937,  78.5983,  57.7471],
        [ 23.6421,  44.4573,  29.1521,  ...,  47.4170,  39.9478,  39.4194],
  

Métriques avec la méthode PyTorch Dropout Before

In [ ]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance_predicted, variance_max, predictive_entropy_predicted, predictive_entropy_max, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(
    model_beforeTorch, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")

for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")

    metric_result = metric_values[metric]

    if metric == "variance_predicted":
        print(f"Variance softmax moyenne : {metric_values['variance_predicted_mean']:.6f}")
        print(f"Variance softmax par échantillon :\n{metric_result}\n")
    elif metric == "variance_max":
        print(f"Variance max moyenne : {metric_values['variance_max_mean']:.6f}")
        print(f"Variance max par échantillon :\n{metric_result}\n")
    elif metric == "predictive_entropy_predicted":
        print(f"Entropie prédictive moyenne : {metric_values['predictive_entropy_predicted_mean']:.6f}")
        print(f"Entropie prédictive par échantillon :\n{metric_result}\n")
    elif metric == "predictive_entropy_max":
        print(f"Entropie max moyenne : {metric_values['predictive_entropy_max_mean']:.6f}")
        print(f"Entropie max par échantillon :\n{metric_result}\n")
    elif metric == "relative_norm":
        print(f"Norme relative moyenne : {metric_values['relative_norm_mean']:.6f}")
        print(f"Norme relative par échantillon :\n{metric_result}\n")
    else:
        print(f"Résultat : {metric_result}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 21.40 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.005692 s
Résultat : tensor([[0.0614, 0.0406, 0.0492,  ..., 0.0106, 0.1701, 0.0296],
        [0.0546, 0.3134, 0.0112,  ..., 0.0047, 0.5267, 0.0162],
        [0.1712, 0.1051, 0.0672,  ..., 0.0289, 0.3531, 0.0580],
        ...,
        [0.3508, 0.1010, 0.2363,  ..., 0.0027, 0.0989, 0.0721],
        [0.2390, 0.0143, 0.1340,  ..., 0.0117, 0.2989, 0.0146],
        [0.0297, 0.0089, 0.0627,  ..., 0.0432, 0.0214, 0.0328]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.012207 s
Variance moyenne : 16.811264
Variance par échantillon :
tensor([[10.4430, 16.1818,  8.1745,  ..., 13.0734, 19.4507, 22.3220],
        [11.6696, 25.8104, 15.0101,  ..., 30.5777, 22.8480, 13.8479],
        [ 6.0730, 15.0156,  8.0995,  ..., 11.4827, 11.7528, 10.6745],
        ...,
       

Métriques avec la méthode PyTorch Dropout After

In [ ]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance_predicted, variance_max, predictive_entropy_predicted, predictive_entropy_max, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(
    model_afterTorch, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")

for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")

    metric_result = metric_values[metric]

    if metric == "variance_predicted":
        print(f"Variance softmax moyenne : {metric_values['variance_predicted_mean']:.6f}")
        print(f"Variance softmax par échantillon :\n{metric_result}\n")
    elif metric == "variance_max":
        print(f"Variance max moyenne : {metric_values['variance_max_mean']:.6f}")
        print(f"Variance max par échantillon :\n{metric_result}\n")
    elif metric == "predictive_entropy_predicted":
        print(f"Entropie prédictive moyenne : {metric_values['predictive_entropy_predicted_mean']:.6f}")
        print(f"Entropie prédictive par échantillon :\n{metric_result}\n")
    elif metric == "predictive_entropy_max":
        print(f"Entropie max moyenne : {metric_values['predictive_entropy_max_mean']:.6f}")
        print(f"Entropie max par échantillon :\n{metric_result}\n")
    elif metric == "relative_norm":
        print(f"Norme relative moyenne : {metric_values['relative_norm_mean']:.6f}")
        print(f"Norme relative par échantillon :\n{metric_result}\n")
    else:
        print(f"Résultat : {metric_result}\n")